In [ ]:
import os, random
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
tf.random.set_seed(SEED)
import torch
torch.manual_seed(SEED)

In [ ]:
# 100,000 features (DPD)
df1 = pd.read_csv('diabetes_dataset_with_notes.csv')


In [ ]:
df1.head()

In [ ]:

df1 = df1.drop(columns=['clinical_notes', 'location', 'year',
                        'race:AfricanAmerican', 'race:Asian', 'race:Caucasian',
                        'race:Hispanic', 'race:Other'])

# Gender: drop the 18 'Other' rows instead of relabelling them as Female\Male
print(df1['gender'].value_counts())
df1 = df1[df1['gender'] != 'Other'].copy()

# Smoking history: lowercase FIRST, then map using lowercase keys
df1['smoking_history'] = (df1['smoking_history']
                          .str.lower().str.strip()
                          .replace({'no info': 'unknown',
                                    'not current': 'former',
                                    'ever': 'former'}))
print(df1['smoking_history'].value_counts())   # should show: unknown, never, former, current

# Duplicates: remove exact duplicate rows so copies can't sit in both train and test
n_before = len(df1)
df1 = df1.drop_duplicates().reset_index(drop=True)
print(f"Removed {n_before - len(df1)} duplicate rows; {len(df1)} remain")
print(df1['diabetes'].value_counts(normalize=True))

In [ ]:
pd.DataFrame({
    "DType": df1.dtypes,
    "null_ratio": df1.isnull().sum(),
    "nunique": df1.nunique()
})

In [ ]:
df1['gender'] = df1['gender'].map({'Female': 0, 'Male': 1})

df1 = pd.get_dummies(df1, columns=['smoking_history'], prefix='smoking', dtype=int)

pd.DataFrame({"DType": df1.dtypes,
              "n_null": df1.isnull().sum(),
              "nunique": df1.nunique()})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

rates = pd.concat({c: df1.groupby(c)['diabetes'].mean()
                   for c in ['gender', 'hypertension', 'heart_disease']})
rates.plot(kind='bar', ax=axes[0], title='Diabetes rate by category')
axes[0].set_ylabel('Proportion diabetic')

sns.boxplot(x='diabetes', y='hbA1c_level', data=df1, ax=axes[1])
axes[1].set_title('HbA1c vs diabetes')

sns.boxplot(x='diabetes', y='blood_glucose_level', data=df1, ax=axes[2])
axes[2].set_title('Blood glucose vs diabetes')

plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(5,7))
sns.countplot(x='gender', hue='diabetes', data=df1, palette='Set2')
plt.title("Gender vs Diabetes")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.legend(title="Diabetes", labels=["0/No", "1/Yes"])
plt.grid(axis='y',linestyle='--')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X = df1.drop(columns=['diabetes'])
y = df1['diabetes']
feature_names = X.columns.tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

# Reset indices so .iloc in the CV loops lines up with the numpy arrays
y_train, y_val, y_test = [s.reset_index(drop=True) for s in (y_train, y_val, y_test)]

for name, s in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(name, len(s), "diabetic rate:", round(s.mean(), 4))

In [ ]:
def count_outliers(df, columns):
    out = {}
    for col in columns:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        out[col] = int(((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum())
    return out

# Report only: HbA1c/glucose "outliers" are real diabetic readings, so they are NOT capped
print("IQR outliers (train):",
      count_outliers(X_train, ['age', 'bmi', 'hbA1c_level', 'blood_glucose_level']))

# Cap BMI only, using bounds from the training set
q1, q3 = X_train['bmi'].quantile([0.25, 0.75])
iqr = q3 - q1
bmi_low, bmi_high = q1 - 1.5*iqr, q3 + 1.5*iqr
for d in (X_train, X_val, X_test):
    d['bmi'] = d['bmi'].clip(bmi_low, bmi_high)
print(f"BMI capped to [{bmi_low:.2f}, {bmi_high:.2f}]")

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Step 1: Initialize scaler
scaler = MinMaxScaler()

# Step 2: Fit on TRAINING data (before oversampling)
X_train = scaler.fit_transform(X_train)

# Step 3: Transform validation and test data
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
# Check min and max values
print("TRAIN MIN:", X_train.min())
print("TRAIN MAX:", X_train.max())

print("\nVALIDATION MIN:", X_val.min())
print("VALIDATION MAX:", X_val.max())

print("\nTEST MIN:", X_test.min())
print("TEST MAX:", X_test.max())

In [ ]:
binary_idx = [feature_names.index(c) for c in ['gender', 'hypertension', 'heart_disease']]
smoke_idx  = [i for i, c in enumerate(feature_names) if c.startswith('smoking_')]

def fix_synthetic(X):
    X = np.asarray(X, dtype=float).copy()
    X[:, binary_idx] = np.round(X[:, binary_idx])
    s = X[:, smoke_idx]
    onehot = np.zeros_like(s)
    onehot[np.arange(len(s)), s.argmax(axis=1)] = 1
    X[:, smoke_idx] = onehot
    return X

In [ ]:
# Run once if any of these are missing
# %pip install smote-variants pytorch-tabnet xgboost catboost shap

In [ ]:

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)
import matplotlib.pyplot as plt

# ProWSyn will be applied inside each training fold during cross-validation.
X_train_meta = X_train
y_train_meta = y_train
X_test_meta = X_test
y_test_meta = y_test

print(f"Training data shape: {X_train_meta.shape}")
print(f"Training labels shape: {y_train_meta.shape}")
print(f"Test data shape: {X_test_meta.shape}")
print(f"Test labels shape: {y_test_meta.shape}")

In [ ]:
# CELL: Cross-validation setup (shared by TabNet, ResNet and XGBoost)

import smote_variants as sv
from collections import Counter
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (classification_report, roc_auc_score,
                             average_precision_score, precision_recall_curve)

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


def best_threshold(y_true, prob):
    """Threshold that maximises F1 on the validation set."""
    p, r, t = precision_recall_curve(y_true, prob)
    f1 = 2 * p * r / (p + r + 1e-12)
    return t[np.argmax(f1[:-1])]

In [ ]:
# CELL: TabNet Training with Cross-Validation

from pytorch_tabnet.tab_model import TabNetClassifier
import torch

# Initialize OOF, test and validation prediction arrays
oof_tabnet = np.zeros(len(X_train_meta))
test_tabnet = np.zeros(len(X_test_meta))
val_tabnet = np.zeros(len(X_val))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_meta, y_train_meta), 1):
    print("\n" + "=" * 60)
    print(f"TABNET FOLD {fold}/{N_FOLDS}")
    print("=" * 60)

    # Split data
    X_fold_train = X_train_meta[train_idx]
    y_fold_train = y_train_meta.iloc[train_idx].copy()

    X_fold_val = X_train_meta[val_idx]

    # Early-stopping set taken from the TRAINING part of the fold,
    # so the OOF rows (X_fold_val) are never seen during training
    X_fold_train, X_es, y_fold_train, y_es = train_test_split(
        X_fold_train, y_fold_train, test_size=0.1,
        stratify=y_fold_train, random_state=SEED)

    # Apply ProWSyn ONLY to the training fold
    oversampler = sv.ProWSyn(
        n_neighbors=3,
        random_state=SEED
    )

    X_fold_train, y_fold_train = oversampler.sample(
        np.asarray(X_fold_train),
        np.asarray(y_fold_train)
    )
    X_fold_train = fix_synthetic(X_fold_train)

    print("Training class distribution:", Counter(y_fold_train))
    print(f"Early-stopping set: {X_es.shape}")
    print(f"Fold validation (OOF): {X_fold_val.shape}")

    # Build model
    tabnet_model = TabNetClassifier(
        n_d=16,
        n_a=16,
        n_steps=7,
        gamma=1.5,
        lambda_sparse=1e-2,
        optimizer_fn=torch.optim.AdamW,
        optimizer_params=dict(lr=0.005),
        seed=SEED + fold,
        verbose=1
    )

    # Train
    tabnet_model.fit(
        X_fold_train,
        y_fold_train,
        eval_set=[(X_es, np.asarray(y_es))],
        eval_name=['es'],
        eval_metric=['auc'],
        max_epochs=50,
        patience=5,
        batch_size=1024,
        virtual_batch_size=128
    )

    # OOF prediction
    oof_tabnet[val_idx] = tabnet_model.predict_proba(X_fold_val)[:, 1]

    # Test and validation predictions (averaged over folds)
    test_tabnet += tabnet_model.predict_proba(X_test_meta)[:, 1] / N_FOLDS
    val_tabnet += tabnet_model.predict_proba(X_val)[:, 1] / N_FOLDS

print("\nTabNet OOF, validation and test predictions completed!")

In [ ]:
tabnet_thr = best_threshold(y_val, val_tabnet)          # chosen on VALIDATION
tabnet_pred = (test_tabnet >= tabnet_thr).astype(int)   # applied to TEST (fold-averaged)

print(f"TabNet Results (threshold = {tabnet_thr:.3f}, chosen on validation)")
print(classification_report(y_test, tabnet_pred, digits=4))
print("ROC-AUC:", round(roc_auc_score(y_test, test_tabnet), 4))
print("PR-AUC:", round(average_precision_score(y_test, test_tabnet), 4))

In [ ]:
# CELL: ResNet (tabular, with residual connections) Training with Cross-Validation

from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import AdamW


def build_resnet(n_features, width=64, n_blocks=3, dropout=0.2):
    inp = layers.Input(shape=(n_features,))
    x = layers.Dense(width)(inp)
    for _ in range(n_blocks):
        h = layers.BatchNormalization()(x)
        h = layers.Dense(width * 2, activation='relu')(h)
        h = layers.Dropout(dropout)(h)
        h = layers.Dense(width)(h)
        x = layers.Add()([x, h])          # skip (residual) connection
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return Model(inp, out)


# Initialize OOF, test and validation prediction arrays
oof_resnet = np.zeros(len(X_train_meta))
test_resnet = np.zeros(len(X_test_meta))
val_resnet = np.zeros(len(X_val))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_meta, y_train_meta), 1):
    print("\n" + "=" * 60)
    print(f"RESNET FOLD {fold}/{N_FOLDS}")
    print("=" * 60)

    # Split data
    X_fold_train = X_train_meta[train_idx]
    y_fold_train = y_train_meta.iloc[train_idx].copy()

    X_fold_val = X_train_meta[val_idx]

    # Early-stopping set taken from the TRAINING part of the fold,
    # so the OOF rows (X_fold_val) are never seen during training
    X_fold_train, X_es, y_fold_train, y_es = train_test_split(
        X_fold_train, y_fold_train, test_size=0.1,
        stratify=y_fold_train, random_state=SEED)

    # Apply ProWSyn ONLY to the training fold
    oversampler = sv.ProWSyn(
        n_neighbors=3,
        random_state=SEED
    )

    X_fold_train, y_fold_train = oversampler.sample(
        np.asarray(X_fold_train),
        np.asarray(y_fold_train)
    )
    X_fold_train = fix_synthetic(X_fold_train)

    print("Training class distribution:", Counter(y_fold_train))
    print(f"Early-stopping set: {X_es.shape}")
    print(f"Fold validation (OOF): {X_fold_val.shape}")

    # Build model
    resnet_model = build_resnet(X_fold_train.shape[1])

    resnet_model.compile(
        optimizer=AdamW(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    early_stop = EarlyStopping(
        monitor='val_loss',
        mode='min',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )

    history = resnet_model.fit(
        X_fold_train,
        y_fold_train,
        validation_data=(X_es, y_es),
        epochs=100,
        batch_size=1024,
        callbacks=[early_stop],
        verbose=1
    )

    # OOF prediction
    oof_resnet[val_idx] = resnet_model.predict(X_fold_val, verbose=0).ravel()

    # Test and validation predictions (averaged over folds)
    test_resnet += resnet_model.predict(X_test_meta, verbose=0).ravel() / N_FOLDS
    val_resnet += resnet_model.predict(X_val, verbose=0).ravel() / N_FOLDS

print("\nResNet OOF, validation and test predictions completed!")

In [ ]:
resnet_thr = best_threshold(y_val, val_resnet)          # chosen on VALIDATION
resnet_pred = (test_resnet >= resnet_thr).astype(int)   # applied to TEST (fold-averaged)

print(f"ResNet Results (threshold = {resnet_thr:.3f}, chosen on validation)")
print(classification_report(y_test, resnet_pred, digits=4))
print("ROC-AUC:", round(roc_auc_score(y_test, test_resnet), 4))
print("PR-AUC:", round(average_precision_score(y_test, test_resnet), 4))

In [ ]:
# CELL: XGBoost base model with Cross-Validation
# Tree models handle the 91/9 imbalance fine, so this one trains on the ORIGINAL fold data (no ProWSyn).
# It sees the data differently from the neural nets, which is what the ensemble needs.

from xgboost import XGBClassifier

oof_xgb = np.zeros(len(X_train_meta))
test_xgb = np.zeros(len(X_test_meta))
val_xgb = np.zeros(len(X_val))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_meta, y_train_meta), 1):
    xgb_base = XGBClassifier(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=SEED
    )
    xgb_base.fit(X_train_meta[train_idx], y_train_meta.iloc[train_idx])

    oof_xgb[val_idx] = xgb_base.predict_proba(X_train_meta[val_idx])[:, 1]
    test_xgb += xgb_base.predict_proba(X_test_meta)[:, 1] / N_FOLDS
    val_xgb += xgb_base.predict_proba(X_val)[:, 1] / N_FOLDS
    print(f"XGBoost fold {fold}/{N_FOLDS} done")

xgb_thr = best_threshold(y_val, val_xgb)
xgb_base_pred = (test_xgb >= xgb_thr).astype(int)

print(f"XGBoost Results (threshold = {xgb_thr:.3f}, chosen on validation)")
print(classification_report(y_test, xgb_base_pred, digits=4))
print("ROC-AUC:", round(roc_auc_score(y_test, test_xgb), 4))
print("PR-AUC:", round(average_precision_score(y_test, test_xgb), 4))

In [ ]:
# CELL: Final ensemble
# Idea taken from HiTCLe (Shaheen et al., 2024): the meta-model gets the base predictions
# together with the original features, so it can learn where each base model is reliable.
# DNN is left out: it is the weakest model and adds nothing that ResNet/TabNet do not already give.

from catboost import CatBoostClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, classification_report)


def to_logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


final_bases = {
    'TabNet': (oof_tabnet, val_tabnet, test_tabnet),
    'ResNet': (oof_resnet, val_resnet, test_resnet),
    'XGBoost': (oof_xgb, val_xgb, test_xgb),
}

X_meta_final_train = np.column_stack([to_logit(o) for o, v, t in final_bases.values()] + [X_train_meta])
X_meta_final_val = np.column_stack([to_logit(v) for o, v, t in final_bases.values()] + [X_val])
X_meta_final_test = np.column_stack([to_logit(t) for o, v, t in final_bases.values()] + [X_test_meta])

final_meta = CatBoostClassifier(
    iterations=300,
    depth=5,
    learning_rate=0.03,
    loss_function='Logloss',
    verbose=False,
    random_seed=SEED
)
final_meta.fit(X_meta_final_train, y_train_meta)

final_val_prob = final_meta.predict_proba(X_meta_final_val)[:, 1]
final_prob = final_meta.predict_proba(X_meta_final_test)[:, 1]

final_thr = best_threshold(y_val, final_val_prob)       # chosen on VALIDATION
final_pred = (final_prob >= final_thr).astype(int)      # applied to TEST

final_results = pd.DataFrame([{
    'Model': 'CatBoost + features: TabNet + ResNet + XGBoost',
    'Accuracy': accuracy_score(y_test, final_pred),
    'Precision': precision_score(y_test, final_pred),
    'Recall': recall_score(y_test, final_pred),
    'F1': f1_score(y_test, final_pred),
    'ROC-AUC': roc_auc_score(y_test, final_prob),
    'PR-AUC': average_precision_score(y_test, final_prob)
}])

print(f"Final ensemble (threshold = {final_thr:.3f}, chosen on validation)")
print(classification_report(y_test, final_pred, digits=4))
final_results

In [ ]:
# CELL: Ablation - effect of adding XGBoost and of giving the meta-learner the original features

combos = {
    'TabNet + ResNet': ['TabNet', 'ResNet'],
    'TabNet + ResNet + XGBoost': ['TabNet', 'ResNet', 'XGBoost'],
}

rows = []
for label, names in combos.items():
    for with_feat in (False, True):
        def build(i, X_raw):
            cols = [to_logit(final_bases[n][i]) for n in names]
            return np.column_stack(cols + ([X_raw] if with_feat else []))

        tr = build(0, X_train_meta)
        va = build(1, X_val)
        te = build(2, X_test_meta)

        m = CatBoostClassifier(iterations=300, depth=5, learning_rate=0.03,
                               loss_function='Logloss', verbose=False, random_seed=SEED)
        m.fit(tr, y_train_meta)
        pv = m.predict_proba(va)[:, 1]
        pt = m.predict_proba(te)[:, 1]
        pred = (pt >= best_threshold(y_val, pv)).astype(int)

        rows.append({
            'Model': f"CatBoost{' + features' if with_feat else ''}: {label}",
            'Accuracy': accuracy_score(y_test, pred),
            'Precision': precision_score(y_test, pred),
            'Recall': recall_score(y_test, pred),
            'F1': f1_score(y_test, pred),
            'ROC-AUC': roc_auc_score(y_test, pt),
            'PR-AUC': average_precision_score(y_test, pt),
        })

ablation_results = pd.DataFrame(rows)
ablation_results

In [ ]:
# Base model results (fold-averaged test predictions, threshold chosen on validation)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def summarize(name, prob, thr):
    pred = (prob >= thr).astype(int)
    return {'Model': name,
            'Accuracy': accuracy_score(y_test, pred),
            'Precision': precision_score(y_test, pred),
            'Recall': recall_score(y_test, pred),
            'F1': f1_score(y_test, pred),
            'ROC-AUC': roc_auc_score(y_test, prob),
            'PR-AUC': average_precision_score(y_test, prob)}


base_results = pd.DataFrame([
    summarize('TabNet', test_tabnet, tabnet_thr),
    summarize('ResNet', test_resnet, resnet_thr),
    summarize('XGBoost', test_xgb, xgb_thr),
])
base_results

In [ ]:
# Summary of Base Models + Ensembles

all_results = pd.concat([
    base_results,
    ablation_results,
    final_results,
], ignore_index=True)

all_results

In [ ]:
import numpy as np
import shap

# Background samples for Kernel SHAP
background = shap.sample(X_train_meta, 100, random_state=42)

# Test samples to explain
X_sample = shap.sample(X_test_meta, 200, random_state=42)

# Explain probability of the diabetic class (class = 1)
explainer = shap.KernelExplainer(
    lambda x: tabnet_model.predict_proba(x)[:, 1],
    background
)

# Compute SHAP values
shap_values = explainer.shap_values(X_sample)

# Check output
print("SHAP type:", type(shap_values))
print("SHAP shape:", np.array(shap_values).shape)
print("X_sample shape:", X_sample.shape)

In [ ]:
# If SHAP returns a 3D array (samples, features, classes)
if len(np.array(shap_values).shape) == 3:
    shap_diabetes = shap_values[:, :, 1]   # Class 1 = diabetic

# If SHAP returns a list (older SHAP versions)
elif isinstance(shap_values, list):
    shap_diabetes = shap_values[1]

# Already a 2D array
else:
    shap_diabetes = shap_values

# Summary plot
shap.summary_plot(
    shap_diabetes,
    X_sample,
    feature_names=X.columns.tolist()
)

In [ ]:
# Predictions of the final ensemble (threshold chosen on validation)
y_pred = final_pred

print("y_pred shape:", y_pred.shape)
print("First 10 predictions:", y_pred[:10])

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5.5, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Non-Diabetic", "Diabetic"]
)

disp.plot(
    cmap="Blues",
    ax=ax,
    colorbar=False,
    values_format="d"
)

# Improve formatting for IEEE paper
ax.set_title("Confusion Matrix of TaReDNet", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted Class", fontsize=11)
ax.set_ylabel("Actual Class", fontsize=11)

# Increase font size of numbers inside the cells
for text in ax.texts:
    text.set_fontsize(14)
    text.set_fontweight("bold")

plt.tight_layout()

# Save high-resolution figure
plt.savefig(
    "Figure_4_Confusion_Matrix_TaReDNet.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()